# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset, **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells**, using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- License: [Open Data Commons Attribution License](https://opendatacommons.org/licenses/by/1-0/)

This workflow follows best practices for referencing all entities by their Croissant `@id`, as per schema guidelines.

In [ ]:
# Ensure mlcroissant library is installed and import dependencies
!pip install mlcroissant --quiet
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading
Load metadata and records from the Croissant dataset:

In [ ]:
# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access the main metadata object
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'n/a')}")
print(f"License: {getattr(metadata, 'license', 'n/a')}")

## 2. Data Overview
Review available record sets, their fields and column `@id`s defined in the schema.

Each record set, field, and column has a unique Croissant `@id`.

In [ ]:
print("Available record sets in the dataset (by @id):\n")
record_sets = list(dataset.list_record_sets())
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'n/a')}")

if not record_sets:
    print("[Warning] No record sets found in Croissant metadata.")
else:
    # Choose the first for exploration
    main_record_set = record_sets[0]['@id']
    print(f"\nColumns/fields for main record set '@id': {main_record_set}")
    rs_fields = dataset.list_fields(record_set=main_record_set)
    for field in rs_fields:
        print(f"  - Field @id: {field['@id']}, name: {field.get('name', field['@id'])}, dataType: {field.get('dataType', 'n/a')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All Croissant entities are referenced by their `@id`.

**Note:** Large record sets or remote data sources may take a few moments to load.

In [ ]:
# For demo, extract records from all available record sets
dataframes = {}
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Loading records from record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  {len(df)} records loaded.\n")

# Select the main record set for further exploration
if record_sets:
    main_rs_id = record_sets[0]['@id']
    main_df = dataframes[main_rs_id]
    print(main_df.head())
else:
    print("No dataframes available for exploration.")

## 4. Exploratory Data Analysis (EDA)

Apply standard data processing such as filtering, normalization, and grouping. Reference fields using their Croissant `@id` as column names in the DataFrame.

In [ ]:
# Example EDA: choose a numeric field (@id) dynamically if possible
import numpy as np

if record_sets:
    # Find numeric fields in main record set
    field_defs = dataset.list_fields(record_set=main_rs_id)
    numeric_fields = [f['@id'] for f in field_defs if f.get('dataType', '').lower() in ['number','float','integer']]
    print(f"Numeric fields found: {numeric_fields}")
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Take the first
        print(f"Analyzing numeric field: {numeric_field_id}")

        if numeric_field_id in main_df.columns:
            # Try to convert column to numeric (in case strings)
            main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
            threshold = main_df[numeric_field_id].mean()

            filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
            print(f"Filtered records with {numeric_field_id} above mean ({threshold:.2f}): {len(filtered_df)} rows\n")
            
            # Normalize the field
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Try to group by a categorical field
            cat_fields = [f['@id'] for f in field_defs if f.get('dataType','').lower() not in ['number','float','integer']]
            group_field = cat_fields[0] if cat_fields else None
            if group_field and group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
                print(f"\nGrouped mean of {numeric_field_id} by {group_field}:")
                print(grouped_df.head())
            else:
                print("No suitable categorical group field found.")
        else:
            print(f"Field {numeric_field_id} not found in DataFrame columns.")
    else:
        print("No numeric fields found in the record set.")
else:
    print("No main record set loaded.")

## 5. Visualization
Visualize data distributions for the main numeric field using matplotlib.

If grouping was performed, show the grouped means.

In [ ]:
# Visualization section
if record_sets and numeric_fields:
    # Histogram of the main numeric field
    fig, ax = plt.subplots(figsize=(7,4))
    main_df[numeric_field_id].hist(ax=ax, bins=30, color='skyblue', edgecolor='k')
    ax.set_title(f"Distribution of {numeric_field_id}")
    ax.set_xlabel(numeric_field_id)
    ax.set_ylabel('Frequency')
    plt.show()

    # If grouped_df exists, show a bar chart
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(8,4))
        plt.bar(grouped_df[group_field].astype(str), grouped_df[numeric_field_id], color='coral')
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion

- We loaded dataset metadata and explored available record sets and fields (by Croissant `@id`).
- We extracted records into pandas DataFrames and performed basic EDA including filtering and normalization on a numeric field.
- Visualizations revealed the distribution of key numeric attributes in the data.

This notebook can serve as a template for rapid Croissant-compatible dataset exploration using `mlcroissant`.

For further domain-specific analyses or in-depth reports, consider joining with biological metadata and reviewing the provided field documentation in the schema.